In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
# Path
PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from utils.data_loading import load_data as load_v1_original, load_test_data
from utils.data_loadingV2 import load_data as load_v2_temporal
from utils.data_loadingV3_random import load_data as load_v3_random
from utils.data_loadingV4_file import load_data as load_v4_file
from utils.simple_lds_loss import SimpleLDSLoss
from utils.dir_lds_loss import DIR_LDS_Loss
from models.vanilla_rnn import VanillaRNN

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

In [3]:
CONFIG = {
    'TRAIN_PATH': PROJECT_ROOT / "datasets_challenge" / "train",
    'VAL_RATIO': 0.2,
    'BATCH_SIZE': 64,
    'COLUMNS_TO_DROP': ['sideSlip', 'vyCG', 'longitude', 'latitude', 'time'],
    'SEQUENCE_LENGTH': 20,
    'THRESHOLD': True,
    'THRESHOLD_VALUE': 3.0,
    'THRESHOLD_COLUMN': 'vxCG',
    'TARGET': 'sideSlip',
    'SCALING': True,
    'SEED': 42,
    'NUM_BINS': 100,
    'DIST_CSV': PROJECT_ROOT / "datasets_challenge" / "sideSlip_distribution.csv",
}

In [4]:
print("Loading V1: Original Split")
train_v1, val_v1, pipeline_v1 = load_v1_original(
    train_path=CONFIG['TRAIN_PATH'],
    train_ratio=0.8,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold=CONFIG['THRESHOLD'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    batchsize=CONFIG['BATCH_SIZE'],
    seed=CONFIG['SEED']
)

print("\nLoading V2: Temporal Split")
train_v2, val_v2, scaler_v2, _, _ = load_v2_temporal(
    train_path=CONFIG['TRAIN_PATH'],
    val_ratio=0.15,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    batchsize=CONFIG['BATCH_SIZE'],
    seed=CONFIG['SEED']
)

print("\nLoading V3: Random Split")
train_v3, val_v3, scaler_v3, _, _ = load_v3_random(
    train_path=CONFIG['TRAIN_PATH'],
    val_ratio=0.2,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    batchsize=CONFIG['BATCH_SIZE'],
    seed=CONFIG['SEED']
)

print("\nLoading V4: File-Based Split (File 7)")
train_v4, val_v4, scaler_v4, _, _ = load_v4_file(
    train_path=CONFIG['TRAIN_PATH'],
    val_file_num=17,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    batchsize=CONFIG['BATCH_SIZE'],
    seed=CONFIG['SEED']
)

def extract_targets(loader):
    targets = []
    for _, y in loader:
        targets.append(y)
    return torch.cat(targets).numpy().flatten()

train_targets_v1 = extract_targets(train_v1)
val_targets_v1 = extract_targets(val_v1)

train_targets_v2 = extract_targets(train_v2)
val_targets_v2 = extract_targets(val_v2)

train_targets_v3 = extract_targets(train_v3)
val_targets_v3 = extract_targets(val_v3)

train_targets_v4 = extract_targets(train_v4)
val_targets_v4 = extract_targets(val_v4)

print(f"\nDataset Sizes:")
print(f"V1 Original: Train {len(train_targets_v1):,} | Val {len(val_targets_v1):,}")
print(f"V2 Temporal: Train {len(train_targets_v2):,} | Val {len(val_targets_v2):,}")
print(f"V3 Random:   Train {len(train_targets_v3):,} | Val {len(val_targets_v3):,}")
print(f"V4 File:     Train {len(train_targets_v4):,} | Val {len(val_targets_v4):,}")

Loading V1: Original Split
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_001.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_002.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_003.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_004.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_005.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_006.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_007.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_008.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_009.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_010.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_011.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_012.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_013.csv
c:\mait_g2_drift_challenge\datasets_challenge\trai

In [5]:
def train_eval_with_lds(
    train_loader,
    val_loader,
    train_targets,
    config,
    epochs=30
):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    #  Model
    model = VanillaRNN(
        input_size=22,
        hidden_size=64,
        num_layers=1,
        dropout=0.0
    ).to(device)

    #  LDS loss (TRAIN TARGETS ONLY)
    criterion = SimpleLDSLoss(
    train_targets=train_targets,
    base='smoothl1',
    smoothl1_beta=0.1,
    reweight='sqrt_inv',
    lds=True,
    num_bins=config['NUM_BINS']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)


    # Training
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device).float()

            optimizer.zero_grad()

            preds = model(x).squeeze(-1)
            loss = criterion(preds, y)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        print(
            f"Epoch {epoch:02d} | "
            f"Train LDS Loss: {epoch_loss / len(train_loader):.4f}"
        )

    # Validation (plain MAE)
    model.eval()
    abs_errors = []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device).float()

            preds = model(x).squeeze(-1)
            abs_errors.append(torch.abs(preds - y))

    val_mae = torch.cat(abs_errors).mean().item()

    return model, val_mae


In [6]:
trained_models = {}
lds_results = {}

print("\n LDS on V1 (Original split)")
trained_models['V1'], lds_results['V1'] = train_eval_with_lds(
    train_v1, val_v1, train_targets_v1, CONFIG
)

print("\n LDS on V2 (Temporal split)")
trained_models['V2'], lds_results['V2'] = train_eval_with_lds(
    train_v2, val_v2, train_targets_v2, CONFIG
)

print("\n LDS on V3 (Random split)")
trained_models['V3'], lds_results['V3'] = train_eval_with_lds(
    train_v3, val_v3, train_targets_v3, CONFIG
)

print("\n LDS on V4 (File-based split)")
trained_models['V4'], lds_results['V4'] = train_eval_with_lds(
    train_v4, val_v4, train_targets_v4, CONFIG
)



 LDS on V1 (Original split)
✅ LDS enabled: gaussian kernel (ks=5, sigma=2.0)

SimpleLDSLoss Configuration
Base loss: smoothl1 (beta=0.1)
Reweight: sqrt_inv
Bins: 100
Weight range: [0.021, 5.437]
Weight mean: 1.000 (normalized to 1.0)

Epoch 00 | Train LDS Loss: 0.0191
Epoch 01 | Train LDS Loss: 0.0115
Epoch 02 | Train LDS Loss: 0.0098
Epoch 03 | Train LDS Loss: 0.0091
Epoch 04 | Train LDS Loss: 0.0085
Epoch 05 | Train LDS Loss: 0.0079
Epoch 06 | Train LDS Loss: 0.0077
Epoch 07 | Train LDS Loss: 0.0075
Epoch 08 | Train LDS Loss: 0.0072
Epoch 09 | Train LDS Loss: 0.0071
Epoch 10 | Train LDS Loss: 0.0069
Epoch 11 | Train LDS Loss: 0.0069
Epoch 12 | Train LDS Loss: 0.0067
Epoch 13 | Train LDS Loss: 0.0066
Epoch 14 | Train LDS Loss: 0.0065
Epoch 15 | Train LDS Loss: 0.0065
Epoch 16 | Train LDS Loss: 0.0064
Epoch 17 | Train LDS Loss: 0.0063
Epoch 18 | Train LDS Loss: 0.0064
Epoch 19 | Train LDS Loss: 0.0062
Epoch 20 | Train LDS Loss: 0.0062
Epoch 21 | Train LDS Loss: 0.0062
Epoch 22 | Train

In [7]:
print("\nFinal Comparison (Validation MAE)")
for k, v in lds_results.items():
    print(f"{k}: MAE = {v:.4f}")



Final Comparison (Validation MAE)
V1: MAE = 0.1535
V2: MAE = 0.1487
V3: MAE = 0.2908
V4: MAE = 0.2686
